# FlowCyt: from twelve marker channels to eight population gates

The scientific task is to estimate six cell-population fractions for each patient. Individual cells have twelve measured channels, but the downstream system requires a small histogram. This notebook exposes the fast fixture workflow step by step: data audit, patient split, classifier posteriors, mixture scores, ScoreQuant, bin templates, and the final count likelihood.

## 1. Load data with patient provenance

The committed fixture is a small integration analogue of the 600,000-cell study. Every row retains its expert class, patient, and original row number. Patient identities—not random cells—define the reference and held-out cohorts, preventing patient leakage.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples.cell_population import (
    CLASS_NAMES,
    FEATURE_NAMES,
    REFERENCE_PATIENTS,
    TEST_PATIENTS,
    load_fixture,
)
from examples.cell_population.closure import (
    fixed_total_partition_audit,
    template_identifiability_audit,
)
from examples.cell_population.likelihood import estimate_bin_templates, fit_binned_mixture
from examples.cell_population.scores import (
    fit_score_model,
    integration_weights,
    reference_composition,
)

data = load_fixture(Path("examples/data/flowcyt_fixture.npz"))
data.features.shape, len(np.unique(data.patients)), len(FEATURE_NAMES)

In [ ]:
patients = np.unique(data.patients)
compositions = np.asarray(
    [
        np.bincount(data.labels[data.patients == patient], minlength=len(CLASS_NAMES))
        for patient in patients
    ]
)
compositions = compositions / compositions.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(11, 4))
bottom = np.zeros(len(patients))
for index, name in enumerate(CLASS_NAMES):
    ax.bar(patients, compositions[:, index], bottom=bottom, label=name)
    bottom += compositions[:, index]
ax.set(
    xlabel="patient",
    ylabel="expert-label fraction",
    title="Class balance varies substantially by patient",
)
ax.legend(ncols=3, fontsize=8);

In [ ]:
reference = data.patients_in(REFERENCE_PATIENTS)
held_out = data.patients_in(TEST_PATIENTS)
len(reference.labels), len(held_out.labels), sorted(np.unique(held_out.patients).tolist())

## 2. Build scores without putting classifier training in ScoreQuant

The application fits a patient-cross-fitted multiclass classifier using reference patients only. Its calibrated posteriors become component density ratios through `ratios_from_posteriors`, and `mixture_scores_from_ratios` turns those ratios into five independent mixture-fraction scores. The classifier is one estimator of density ratios, so its closure must be audited separately from compression loss.

In [ ]:
score_fit = fit_score_model(reference, max_per_patient_class=128, max_iter=20, seed=2026)
theta0 = reference_composition(reference.labels, reference.patients)
reference_scores = np.asarray(
    sq.mixture_scores_from_ratios(
        sq.ratios_from_posteriors(
            score_fit.out_of_fold_probabilities, score_fit.model.class_priors
        ),
        theta0,
    )
)
held_out_posteriors = score_fit.model.predict_proba(held_out.features)
held_out_scores = np.asarray(
    sq.mixture_scores_from_ratios(
        sq.ratios_from_posteriors(held_out_posteriors, score_fit.model.class_priors), theta0
    )
)
score_fit.calibration_selection["selected_strategy"], theta0, reference_scores.shape

## 3. Learn eight hard bins through the public API

Reference rows receive deterministic application roles: partition fitting, validation diagnostics, or template estimation. The validation rows cannot affect optimization. Expert labels are needed by the application to define reference integration weights and later estimate $P(B_j\mid k)$; ScoreQuant itself receives only scores and weights.

In [ ]:
role = np.mod(reference.source_rows + 17 * reference.patients + 101 * reference.labels, 4)
partition_mask, validation_mask, template_mask = role <= 1, role == 2, role == 3
partition_weights = integration_weights(
    reference.labels[partition_mask], reference.patients[partition_mask], theta0
)
validation_weights = integration_weights(
    reference.labels[validation_mask], reference.patients[validation_mask], theta0
)
quantizer = sq.fit_quantizer(
    sq.ScoreSample(reference_scores[partition_mask], partition_weights),
    validation=sq.ScoreSample(reference_scores[validation_mask], validation_weights),
    n_bins=8,
    criterion=sq.DOptimality(),
    config=sq.SoftVoronoiConfig(seed=2026, n_init=2, max_steps=40, record_every=10),
)
quantizer.report()

## 4. Freeze templates, then fit only bin counts

After the gates are frozen, application code estimates a six-column template matrix from unused reference rows. A held-out patient is reduced to eight integers; the multinomial mixture likelihood sees neither marker values nor expert labels.

In [ ]:
template_bins = np.asarray(quantizer.predict_scores(reference_scores[template_mask]))
templates = estimate_bin_templates(
    reference.labels[template_mask], template_bins, reference.patients[template_mask], n_bins=8
)
held_out_bins = np.asarray(quantizer.predict_scores(held_out_scores))
held_out_patients = np.unique(held_out.patients)
estimates, truth = [], []
for patient in held_out_patients:
    mask = held_out.patients == patient
    counts = np.bincount(held_out_bins[mask], minlength=8)
    estimates.append(fit_binned_mixture(counts, templates).fractions)
    class_counts = np.bincount(held_out.labels[mask], minlength=len(CLASS_NAMES))
    truth.append(class_counts / class_counts.sum())
estimates, truth = np.asarray(estimates), np.asarray(truth)
per_class_rmse = np.sqrt(np.mean((estimates - truth) ** 2, axis=0))
dict(zip(CLASS_NAMES, per_class_rmse, strict=True))

In [ ]:
bin_mass = templates @ theta0
bin_composition = templates * theta0[None, :] / bin_mass[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for index, name in enumerate(CLASS_NAMES[:5]):
    axes[0].scatter(truth[:, index], estimates[:, index], label=name)
limit = max(truth[:, :5].max(), estimates[:, :5].max())
axes[0].plot([0, limit], [0, limit], "k--")
axes[0].set(
    title="Fractions from eight counts", xlabel="expert fraction", ylabel="estimated fraction"
)
axes[0].legend(fontsize=8)
image = axes[1].imshow(bin_composition.T, aspect="auto", vmin=0, vmax=1, cmap="viridis")
axes[1].set(
    title="Reference composition of each bin",
    xlabel="hard bin",
    ylabel="population",
    yticks=range(6),
    yticklabels=CLASS_NAMES,
)
fig.colorbar(image, ax=axes[1], label="P(population | bin)");

## 5. Distinguish compression from downstream identifiability

For fixed-total counts, $B$ bins provide only $B-1$ independent frequencies. Six fractions have five free directions, so at least six bins are necessary. The example-local audits below check both conditional Fisher rank and the rank of the template contrasts; these diagnostics intentionally remain outside the public library API.

In [ ]:
validation_bins = np.asarray(quantizer.predict_scores(reference_scores[validation_mask]))
fixed_total = fixed_total_partition_audit(
    reference_scores[validation_mask], validation_bins, validation_weights, n_bins=8
)
identifiability = template_identifiability_audit(templates, theta0)
{
    "fixed-total retained rank": fixed_total["retained_rank"],
    "template contrast rank": identifiability["effective_rank"],
    "downstream identifiable": identifiability["full_rank"],
}

## Full frozen evidence

The published study uses 600,000 cells, the frozen raw-declared-prior classifier strategy, and an eight-bin operating point. It separately reports ratio-model closure, fixed-total rank, pseudo-patient bias, seed stability, and boundary-aware uncertainty coverage. The small fixture above teaches the interfaces; it is not the headline measurement. See the FlowCyt evidence chapter for the full protocol and results.